# Data Ingestion Phase
This notebook describes steps required to store, ingest and validate different city datasets

## 0. Downloading Datasets

The raw data file structure is as follows for now:

- `data/raw/taxi_trips/yellow`
- `data/raw/taxi_zones`
- `data/raw/weather`
- `data/raw/air_quality`

Note: we are only going to use data for year 2024 in week 1.

### 0a. Downloading Datasets

Download all datasets from the shared Google Drive folder and arrange them under `data/raw/`.

In [1]:
from pathlib import Path
from shutil import copy2
from zipfile import ZipFile

import gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1qjBtPVDepDE22j0axqrLVR0A2a969Qyy?usp=sharing"

root = Path.cwd().resolve()
if not (root / "data" / "raw").exists():
    root = root.parent

raw = root / "data" / "raw"
download_dir = root / "data" / "drive_download"
download_dir.mkdir(parents=True, exist_ok=True)

# The folder must be shared so that anyone with the link can view/download it.
gdown.download_folder(
    url=DRIVE_FOLDER_URL,
    output=str(download_dir),
    quiet=False,
)

for source in download_dir.rglob("*"):
    if not source.is_file():
        continue

    name = source.name
    if name == "air_quality.zip":
        out_dir = raw / "air_quality"
        out_dir.mkdir(parents=True, exist_ok=True)
        with ZipFile(source) as archive:
            archive.extractall(out_dir)
        print(f"unzip {source.relative_to(root)} -> {out_dir.relative_to(root)}")
    elif name == "taxi_zone_lookup.csv":
        out_dir = raw / "taxi_zones"
        out_dir.mkdir(parents=True, exist_ok=True)
        copy2(source, out_dir / name)
        print(f"copy  {name} -> {out_dir.relative_to(root)}")
    elif name == "weather.csv":
        out_dir = raw / "weather"
        out_dir.mkdir(parents=True, exist_ok=True)
        copy2(source, out_dir / name)
        print(f"copy  {name} -> {out_dir.relative_to(root)}")
    elif name.startswith("yellow_tripdata_") and name.endswith(".parquet"):
        out_dir = raw / "taxi_trips" / "yellow"
        out_dir.mkdir(parents=True, exist_ok=True)
        copy2(source, out_dir / name)
        print(f"copy  {name} -> {out_dir.relative_to(root)}")


Retrieving folder contents


Processing file 1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW air_quality.zip
Processing file 1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t taxi_zone_lookup.csv
Processing file 1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M weather.csv
Processing file 17v0eFEontYEKtGoqyB0v9rj_BEDc7snA yellow_tripdata_2024-01.parquet
Processing file 1N-dRuGdd_lOYGAbdbgJJWMyIsV_lz057 yellow_tripdata_2024-02.parquet
Processing file 1oUxC0cLWqOatddyT8aB06VvFFZY0-lu2 yellow_tripdata_2024-03.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW
From (redirected): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW&confirm=t&uuid=2e5008ba-d0a0-48fc-882e-ee805c2213da
To: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/drive_download/air_quality.zip
100%|██████████| 66.3M/66.3M [00:03<00:00, 18.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t
To: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/drive_download/taxi_zone_lookup.csv
100%|██████████| 12.3k/12.3k [00:00<00:00, 9.62MB/s]
Downloading...
From: https://drive.google.com/uc?id=1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M
To: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/drive_download/weather.csv
100%|██████████| 1.07M/1.07M [00:00<00:00, 14.8MB/s]
Downlo

unzip data/drive_download/air_quality.zip -> data/raw/air_quality
copy  yellow_tripdata_2024-03.parquet -> data/raw/taxi_trips/yellow
copy  taxi_zone_lookup.csv -> data/raw/taxi_zones
copy  weather.csv -> data/raw/weather
copy  yellow_tripdata_2024-02.parquet -> data/raw/taxi_trips/yellow
copy  yellow_tripdata_2024-01.parquet -> data/raw/taxi_trips/yellow


## 1. Loading Datasets into Spark

### 1a. Configure Spark

Create a local Spark session with Delta Lake support.


In [2]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root

spark = create_spark("ingestion-bronze")
ROOT = project_root()
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/samuelflodin/.ivy2/cache
The jars for the packages stored in: /Users/samuelflodin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-73e5d0d7-bd73-4bc5-8001-78ffd559fe07;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 76ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs


### 1b. Partitioning strategy

Each dataset lands as its own Delta table under `data/lake/bronze/`.

| Table | Partition columns | Why |
| --- | --- | --- |
| `taxi_trips` | `taxi_type`, `pickup_date` | Color filter + daily time pruning for trip analytics |
| `weather` | `observation_date` | Hourly series almost always queried by day/range |
| `air_quality` | `state_code`, `measurement_date` | City queries prune to NY (`36`); date helps time joins |
| `taxi_zones` | _(none)_ | Tiny lookup table — partitioning would only add overhead |

Partition columns are derived from source timestamps (or state) **before** the Delta write so Spark can prune on read without rewriting later.


### 1c. Shared helpers

In [3]:
import yaml
from pyspark.sql import DataFrame, functions as F
from pyspark.sql import types as T

from src.lake import BRONZE, RAW, ROOT, write_bronze, show_delta


def show_table(table_name: str, n: int = 3) -> None:
    show_delta(spark, BRONZE / table_name, n)


with open(ROOT / "config" / "compatible_types.yaml") as f:
    _type_names = yaml.safe_load(f)

COMPATIBLE = {
    logical: tuple(getattr(T, name) for name in names)
    for logical, names in _type_names.items()
}


def validate_schema(df: DataFrame, expected: dict, dataset: str) -> dict:
    """Require expected columns; allow extra columns; check types."""
    actual = {f.name: f.dataType for f in df.schema.fields}
    missing = [c for c in expected if c not in actual]
    mismatches = []

    for col, expected_type in expected.items():
        if col not in actual:
            continue
        allowed = COMPATIBLE.get(expected_type)
        if allowed and not isinstance(actual[col], allowed):
            mismatches.append(
                f"{col}: got {actual[col].simpleString()}, expected {expected_type}"
            )

    check = {
        "ok": not missing and not mismatches,
        "missing": missing,
        "type_mismatches": mismatches,
    }
    print(f"[{dataset}] schema ok={check['ok']}")
    if check["missing"]:
        print("  missing:", check["missing"])
    if check["type_mismatches"]:
        print("  mismatches:", check["type_mismatches"])
    if not check["ok"]:
        raise ValueError(f"Schema validation failed for {dataset}")
    return check


### 1d. Ingest `taxi_zones`

Small dimension table — validate lookup columns, write unpartitioned Delta.


In [4]:
taxi_zones_expected = {
    "LocationID": "integer",
    "Borough": "string",
    "Zone": "string",
    "service_zone": "string",
}

taxi_zones_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "taxi_zones"))
)

validate_schema(taxi_zones_raw, taxi_zones_expected, "taxi_zones")
write_bronze(taxi_zones_raw, "taxi_zones", partition_by=[])
show_table("taxi_zones")


[taxi_zones] schema ok=True


26/09/11 14:05:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


taxi_zones: 265 rows @ data/lake/bronze/taxi_zones
+----------+-------+-----------------------+------------+--------------------------+
|LocationID|Borough|Zone                   |service_zone|_ingested_at              |
+----------+-------+-----------------------+------------+--------------------------+
|1         |EWR    |Newark Airport         |EWR         |2026-09-11 12:05:04.724757|
|2         |Queens |Jamaica Bay            |Boro Zone   |2026-09-11 12:05:04.724757|
|3         |Bronx  |Allerton/Pelham Gardens|Boro Zone   |2026-09-11 12:05:04.724757|
+----------+-------+-----------------------+------------+--------------------------+
only showing top 3 rows



### 1e. Ingest `weather`

Meteostat hourly weather CSV. Build a timestamp from `year`, `month`, `day`, and `hour`, then partition by `observation_date`.

In [5]:
weather_expected = {
    "year": "integer",
    "month": "integer",
    "day": "integer",
    "hour": "integer",
    "temp": "double",
    "wspd": "double",
}

weather_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "weather"))
)

validate_schema(weather_raw, weather_expected, "weather")

weather_bronze = weather_raw.withColumn(
    "observation_date",
    F.make_date(F.col("year"), F.col("month"), F.col("day")),
)

write_bronze(weather_bronze, "weather", partition_by=["observation_date"])
show_table("weather")


[weather] schema ok=True


weather: 8784 rows @ data/lake/bronze/weather
+----+-----+---+----+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+------+-----------+----+-----------+----+-----------+----------------+--------------------------+
|year|month|day|hour|temp|temp_source|rhum|rhum_source|prcp|prcp_source|snwd|snwd_source|wdir|wdir_source|wspd|wspd_source|wpgt|wpgt_source|pres  |pres_source|cldc|cldc_source|coco|coco_source|observation_date|_ingested_at              |
+----+-----+---+----+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+------+-----------+----+-----------+----+-----------+----------------+--------------------------+
|2024|3    |15 |0   |13.3|isd_lite   |51  |isd_lite   |0.0 |isd_lite   |NULL|NULL       |170 |isd_lite   |22.3|isd_lite   |NULL|NULL       |1015.2|isd_lite   |6   |isd_lite   |3   |dwd_mosmix |2024-03-15      |2026-09-11 12:05:09.272904|
|2

### 1f. Ingest `air_quality`

EPA hourly PM2.5. Keep New York state rows only (`State Code = 36`) for the NYC platform.

Partition by `state_code` + `measurement_date` so city/time filters prune files on both axes.


In [6]:
air_quality_expected = {
    "State Code": "integer",
    "County Code": "integer",
    "Site Num": "integer",
    "Parameter Name": "string",
    "Date GMT": "string",
    "Time GMT": "string",
    "Sample Measurement": "double",
    "Units of Measure": "string",
}

air_quality_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "air_quality" / "hourly_88101_2024.csv"))
)

validate_schema(air_quality_raw, air_quality_expected, "air_quality")

# select nyc only
air_quality_ny = air_quality_raw.filter(F.col("State Code") == 36)

air_quality_bronze = (
    air_quality_ny
    .withColumn("state_code", F.col("State Code"))
    .withColumn(
        "measurement_date",
        F.to_date(F.col("Date GMT"), "yyyy-MM-dd"),
    )
    # Drop originals so Delta partition columns are unambiguous after name sanitizing.
    .drop("State Code")
)

write_bronze(
    air_quality_bronze,
    "air_quality",
    partition_by=["state_code", "measurement_date"],
)
show_table("air_quality")


[air_quality] schema ok=True


air_quality: 117438 rows @ data/lake/bronze/air_quality
+-----------+--------+--------------+---+--------+---------+-----+------------------------+----------+-------------------+----------+-------------------+------------------+---------------------------+---+-----------+---------+-----------+-----------+----------------------------------------------------------------------------------+----------+-----------+-------------------+----------+----------------+--------------------------+
|County_Code|Site_Num|Parameter_Code|POC|Latitude|Longitude|Datum|Parameter_Name          |Date_Local|Time_Local         |Date_GMT  |Time_GMT           |Sample_Measurement|Units_of_Measure           |MDL|Uncertainty|Qualifier|Method_Type|Method_Code|Method_Name                                                                       |State_Name|County_Name|Date_of_Last_Change|state_code|measurement_date|_ingested_at              |
+-----------+--------+--------------+---+--------+---------+-----+--------------

### 1g. Ingest `taxi_trips`

Yellow TLC parquet files are normalized to one schema, tagged with `taxi_type`, and given a derived `pickup_date` partition column.

In [7]:
yellow_expected = {
    "VendorID": "integer",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "long",
    "trip_distance": "double",
    "PULocationID": "long",
    "DOLocationID": "long",
    "fare_amount": "double",
    "total_amount": "double",
}


def normalize_trips(df: DataFrame, taxi_type: str) -> DataFrame:
    """Align pickup/dropoff names and add partition columns."""
    return (
        df.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
        .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
        .withColumn("taxi_type", F.lit(taxi_type))
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )


yellow_raw = spark.read.parquet(str(RAW / "taxi_trips" / "yellow"))
validate_schema(yellow_raw, yellow_expected, "taxi_trips/yellow")

taxi_trips_bronze = normalize_trips(yellow_raw, "yellow")

write_bronze(
    taxi_trips_bronze,
    "taxi_trips",
    partition_by=["taxi_type", "pickup_date"],
)
show_table("taxi_trips")


[taxi_trips/yellow] schema ok=True


taxi_trips: 9554778 rows @ data/lake/bronze/taxi_trips
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------+-----------+--------------------------+
|VendorID|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|taxi_type|pickup_date|_ingested_at              |
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------+-----------+----------------------

### 1h. Bronze summary

Confirm all four Delta tables exist and report row counts.


In [8]:
tables = ["taxi_zones", "weather", "air_quality", "taxi_trips"]

print("Bronze Delta tables")
print("-" * 40)
for name in tables:
    path = BRONZE / name
    df = spark.read.format("delta").load(str(path))
    parts = sorted({p.name.split("=")[0] for p in path.glob("*/") if "=" in p.name})
    print(f"{name:14} rows={df.count():>12,}  partitions={parts or ['(none)']}")


Bronze Delta tables
----------------------------------------
taxi_zones     rows=         265  partitions=['(none)']
weather        rows=       8,784  partitions=['observation_date']
air_quality    rows=     117,438  partitions=['state_code']
taxi_trips     rows=   9,554,778  partitions=['taxi_type']


## 2. Data Normalization and Standardization

Promote bronze → silver: rename columns to snake_case, normalize timestamps/types,
run dataset-specific transforms, apply basic data-quality checks, and write Delta tables
under `data/lake/silver/`. Rejected rows land in `data/lake/bronze/<dataset>_rejects`.

**Timestamps.** Spark session TZ is UTC, but the analytical clock is **America/New_York**.

| Dataset | Source clock | Silver |
| --- | --- | --- |
| `taxi_trips` | TLC naive local (already NY) | keep wall time; reject pickups outside 2024-01-01 … 2024-06-01 |
| `weather` | ISD `DATE` in UTC | `from_utc_timestamp` → NY local; `observation_date` / `hour` from that |
| `air_quality` | EPA Date/Time GMT (UTC) | build UTC timestamp, then convert to NY local; `measurement_date` / `hour` from that |

Gold joins on local `*_date` + `*_hour` without further timezone conversion.


### 2a. Silver helpers

Shared rename, quality-check, and silver write utilities. Delta writes use `src/lake.py`.


In [9]:
from pyspark.sql.window import Window

from src.lake import SILVER, read_delta, write_silver

LOCAL_TZ = "America/New_York"


def utc_to_local(col):
    """UTC instant → America/New_York wall time (Spark session is UTC)."""
    return F.from_utc_timestamp(col, LOCAL_TZ)


def read_bronze(table_name: str) -> DataFrame:
    return read_delta(spark, BRONZE / table_name)


def rename_columns(df: DataFrame, mapping: dict) -> DataFrame:
    """Rename only columns that exist."""
    out = df
    for src, dst in mapping.items():
        if src in out.columns and src != dst:
            out = out.withColumnRenamed(src, dst)
    return out


def write_rejects(df: DataFrame, table_name: str) -> None:
    path = BRONZE / f"{table_name}_rejects"
    (
        df.withColumn("_rejected_at", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(str(path))
    )


def quality_filter(df: DataFrame, not_null=None, timestamp_order=None, numeric_range=None, timestamp_range=None):
    reasons = []

    for col in not_null or []:
        if col in df.columns:
            reasons.append(F.when(F.col(col).isNull(), F.lit(f"null:{col}")))

    if timestamp_order:
        before, after = timestamp_order
        if before in df.columns and after in df.columns:
            bad = (
                F.col(before).isNotNull()
                & F.col(after).isNotNull()
                & (F.col(before) > F.col(after))
            )
            reasons.append(F.when(bad, F.lit(f"ts_order:{before}>{after}")))

    for col, bounds in (numeric_range or {}).items():
        if col not in df.columns:
            continue
        bad = F.lit(False)
        if bounds.get("min") is not None:
            bad = bad | (F.col(col).isNotNull() & (F.col(col) < F.lit(bounds["min"])))
        if bounds.get("max") is not None:
            bad = bad | (F.col(col).isNotNull() & (F.col(col) > F.lit(bounds["max"])))
        reasons.append(F.when(bad, F.lit(f"range:{col}")))

    for col, bounds in (timestamp_range or {}).items():
        if col not in df.columns:
            continue
        bad = F.lit(False)
        if bounds.get("min") is not None:
            bad = bad | (F.col(col).isNotNull() & (F.col(col) < F.to_timestamp(F.lit(bounds["min"]))))
        if bounds.get("max") is not None:
            bad = bad | (F.col(col).isNotNull() & (F.col(col) >= F.to_timestamp(F.lit(bounds["max"]))))
        reasons.append(F.when(bad, F.lit(f"ts_range:{col}")))

    if not reasons:
        return df, df.limit(0).withColumn("_reject_reason", F.lit(""))

    flagged = df.withColumn("_reject_reason", F.concat_ws(",", *reasons))
    good = flagged.filter(F.col("_reject_reason") == "").drop("_reject_reason")
    rejects = flagged.filter(F.col("_reject_reason") != "")
    return good, rejects


def promote(table_name, df, primary_key, partition_by=None, **dq):
    """DQ filter → keep first row per PK → write silver + rejects."""
    good, rejects = quality_filter(df, **dq)

    pk = [c for c in primary_key if c in good.columns]
    if pk:
        ranked = good.withColumn(
            "_rn",
            F.row_number().over(Window.partitionBy(*pk).orderBy(F.lit(1))),
        )
        dup_rejects = (
            ranked.filter(F.col("_rn") > 1)
            .drop("_rn")
            .withColumn("_reject_reason", F.lit("duplicate_pk"))
        )
        good = ranked.filter(F.col("_rn") == 1).drop("_rn")
        rejects = rejects.unionByName(dup_rejects, allowMissingColumns=True)

    write_rejects(rejects, table_name)
    write_silver(good, table_name, partition_by=partition_by)

    n_good = spark.read.format("delta").load(str(SILVER / table_name)).count()
    n_bad = spark.read.format("delta").load(str(BRONZE / f"{table_name}_rejects")).count()
    print(f"[silver/{table_name}] kept={n_good:,}  rejected={n_bad:,}")


### 2b. Silver `taxi_zones`

Standardize lookup column names; fill blank borough/zone labels.


In [10]:
zones = rename_columns(
    read_bronze("taxi_zones"),
    {
        "LocationID": "location_id",
        "Borough": "borough",
        "Zone": "zone",
        "service_zone": "service_zone",
    },
)

zones = (
    zones.select("location_id", "borough", "zone", "service_zone")
    .withColumn("location_id", F.col("location_id").cast("int"))
    .fillna({"borough": "UNKNOWN", "zone": "UNKNOWN", "service_zone": "UNKNOWN"})
)

promote(
    "taxi_zones",
    zones,
    primary_key=["location_id"],
    not_null=["location_id"],
    partition_by=[],
)


[silver/taxi_zones] kept=265  rejected=0


### 2c. Silver `weather`

Standardize Meteostat fields, build the hourly timestamp, and keep temperature (°C) and wind speed (m/s) with local date/hour columns.

In [11]:
weather = read_bronze("weather")

weather = (
    weather.withColumn(
        "obs_timestamp",
        F.make_timestamp(
            F.col("year"),
            F.col("month"),
            F.col("day"),
            F.col("hour"),
            F.lit(0),
            F.lit(0),
        ),
    )
    .withColumn("observation_date", F.to_date("obs_timestamp"))
    .withColumn("observation_hour", F.hour("obs_timestamp"))
    .select(
        F.lit("72505394728").alias("station_id"),
        "obs_timestamp",
        F.col("temp").cast("double").alias("temperature_c"),
        (F.col("wspd").cast("double") / F.lit(3.6)).alias("wind_speed_ms"),
        "observation_date",
        "observation_hour",
    )
)

promote(
    "weather",
    weather,
    primary_key=["station_id", "obs_timestamp"],
    not_null=["station_id", "obs_timestamp"],
    partition_by=["observation_date"],
)


[silver/weather] kept=8,784  rejected=0


### 2d. Silver `air_quality`

Standardize EPA columns, build a UTC timestamp from Date/Time GMT, convert to
America/New_York, keep NY-scoped rows with typed measures.


In [12]:
aq = rename_columns(
    read_bronze("air_quality"),
    {
        "County_Code": "county_code",
        "Site_Num": "site_num",
        "Parameter_Name": "parameter",
        "Date_GMT": "date_gmt",
        "Time_GMT": "time_gmt",
        "Sample_Measurement": "value",
        "Units_of_Measure": "unit",
        # state_code / measurement_date already exist from bronze
    },
)

# Bronze inferred Time_GMT as a timestamp (wrong calendar day); rebuild UTC from date + clock, then NY local.
aq = (
    aq.withColumn(
        "measurement_timestamp",
        utc_to_local(
            F.to_timestamp(
                F.concat_ws(
                    " ",
                    F.date_format(F.col("date_gmt").cast("date"), "yyyy-MM-dd"),
                    F.date_format(F.col("time_gmt").cast("timestamp"), "HH:mm:ss"),
                ),
                "yyyy-MM-dd HH:mm:ss",
            )
        ),
    )
    .withColumn("measurement_date", F.to_date("measurement_timestamp"))
    .withColumn("measurement_hour", F.hour("measurement_timestamp"))
    .select(
        F.col("state_code").cast("int").alias("state_code"),
        F.col("county_code").cast("int").alias("county_code"),
        F.col("site_num").cast("int").alias("site_num"),
        F.col("parameter").cast("string").alias("parameter"),
        F.col("value").cast("double").alias("value"),
        F.col("unit").cast("string").alias("unit"),
        "measurement_timestamp",
        "measurement_date",
        "measurement_hour",
    )
)

promote(
    "air_quality",
    aq,
    primary_key=["state_code", "county_code", "site_num", "parameter", "measurement_timestamp"],
    not_null=["state_code", "county_code", "site_num", "parameter", "measurement_timestamp"],
    numeric_range={"value": {"min": -50, "max": 1000}},
    partition_by=["measurement_date"],
)


[silver/air_quality] kept=112,838  rejected=4,600


### 2e. Silver `taxi_trips`

Snake_case fact columns, cast money/distance types, derive pickup hour.
TLC timestamps are already America/New_York wall time (naive) — do not shift them.
Reject null PKs, inverted timestamps, pickups outside Jan–May 2024, and out-of-range fare/distance.

In [13]:
trips = rename_columns(
    read_bronze("taxi_trips"),
    {
        "VendorID": "vendor_id",
        "PULocationID": "pickup_location_id",
        "DOLocationID": "dropoff_location_id",
        "passenger_count": "passenger_count",
        "trip_distance": "trip_distance",
        "fare_amount": "fare_amount",
        "tip_amount": "tip_amount",
        "tolls_amount": "tolls_amount",
        "total_amount": "total_amount",
        # pickup_datetime / dropoff_datetime / taxi_type / pickup_date already set in bronze
    },
)

trips = (
    trips.withColumn("pickup_datetime", F.to_timestamp("pickup_datetime"))
    .withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime"))
    .withColumn("pickup_date", F.to_date("pickup_datetime"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .select(
        F.col("taxi_type").cast("string").alias("taxi_type"),
        F.col("vendor_id").cast("int").alias("vendor_id"),
        "pickup_datetime",
        "dropoff_datetime",
        F.col("passenger_count").cast("int").alias("passenger_count"),
        F.col("trip_distance").cast("double").alias("trip_distance"),
        F.col("pickup_location_id").cast("int").alias("pickup_location_id"),
        F.col("dropoff_location_id").cast("int").alias("dropoff_location_id"),
        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("tip_amount").cast("double").alias("tip_amount"),
        F.col("tolls_amount").cast("double").alias("tolls_amount"),
        F.col("total_amount").cast("double").alias("total_amount"),
        "pickup_date",
        "pickup_hour",
    )
)

promote(
    "taxi_trips",
    trips,
    primary_key=[
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "pickup_location_id",
        "dropoff_location_id",
    ],
    not_null=["pickup_datetime", "dropoff_datetime", "pickup_location_id", "dropoff_location_id"],
    timestamp_order=("pickup_datetime", "dropoff_datetime"),
    timestamp_range={"pickup_datetime": {"min": "2024-01-01", "max": "2024-06-01"}},
    numeric_range={
        "fare_amount": {"min": 0, "max": 500},
        "trip_distance": {"min": 0, "max": 200},
    },
    partition_by=["pickup_date"],
)


[silver/taxi_trips] kept=9,417,383  rejected=137,395


### 2f. Silver summary


In [14]:
print("Silver Delta tables")
print("-" * 40)
for name in ["taxi_zones", "weather", "air_quality", "taxi_trips"]:
    path = SILVER / name
    df = spark.read.format("delta").load(str(path))
    rejects_path = BRONZE / f"{name}_rejects"
    rejected = 0
    if rejects_path.exists():
        rejected = spark.read.format("delta").load(str(rejects_path)).count()
    print(f"{name:14} silver={df.count():>12,}  rejects={rejected:>10,}")
    df.printSchema()


Silver Delta tables
----------------------------------------


taxi_zones     silver=         265  rejects=         0
root
 |-- location_id: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

weather        silver=       8,784  rejects=         0
root
 |-- station_id: string (nullable = true)
 |-- obs_timestamp: timestamp (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- observation_hour: integer (nullable = true)

air_quality    silver=     112,838  rejects=     4,600
root
 |-- state_code: integer (nullable = true)
 |-- county_code: integer (nullable = true)
 |-- site_num: integer (nullable = true)
 |-- parameter: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- measurement_timestamp: timestamp (nullable = true)
 |-- measurement_date: date (nullable = true)
 |-- measurement_hour: integer (null

26/09/11 18:03:06 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 400838 ms exceeds timeout 120000 ms
26/09/11 18:03:07 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/11 18:03:15 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o